<a href="https://colab.research.google.com/github/NagaRaju1991/google_colab_notebooks/blob/fsds_course/01_BatchNormalization_DropOut_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ──────────────────────────────────────────────────────────────
# Regression: California Housing Prices with nn.Sequential
# ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# ─── 1. LOAD AND PREPARE DATA ─────────────────────────────────────
data = fetch_california_housing()
X = data.data  # 8 features: MedInc, HouseAge, AveRooms, etc.
y = data.target # Target: Median house value in $100,000s

# Split 1: Separate a "blind" Test set (20%) from the rest
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split 2: Divide remaining data into Training (90%) and Validation (10%)
# This 3-way split ensures we don't "overfit" our architecture to the test data.
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.125, random_state=42
)

# Standardize: Essential for MLPs so all features have Mean=0 and Var=1.
# This prevents high-magnitude features (like Population) from dominating gradients.
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train) # Fit and transform on Train
X_val = scaler_X.transform(X_val)         # ONLY transform Val/Test (prevent data leakage)
X_test = scaler_X.transform(X_test)

# Convert to Tensors: PyTorch requires float32 for weights/inputs
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1) # view(-1,1) makes it a column vector
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# DataLoaders: Shuffling Train prevents the model from learning the order of rows.
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)

# ─── 2. DEFINE MODEL WITH nn.Sequential ───────────────────────────
# Sequential chains layers: Output of one is Input of next.
model = nn.Sequential(
    nn.Linear(8, 128),     # Input (8) -> Hidden (128)
    nn.ReLU(),             # Non-linear activation
    nn.BatchNorm1d(128),   # Normalizes activations (xi) to stabilize training
    nn.Dropout(0.15),      # Randomly shuts off 15% of neurons to prevent memorization

    nn.Linear(128, 64),    # Hidden (128) -> Hidden (64)
    nn.ReLU(),
    nn.BatchNorm1d(64),
    nn.Dropout(0.1),       # Lower dropout in deeper layers is common

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.BatchNorm1d(32),

    nn.Linear(32, 1)       # Output: Single neuron for regression (no activation)
)

# Move model to GPU if available for faster math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ─── 3. LOSS & OPTIMIZER ──────────────────────────────────────────
criterion = nn.MSELoss() # Mean Squared Error: Standard for regression tasks
# AdamW: Decouples weight decay from the learning rate for better generalization
optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-5)

# ─── 4. TRAINING LOOP ─────────────────────────────────────────────
epochs = 120
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train() # Enable Dropout and BatchNorm tracking
    train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()       # Clear old gradients from previous batch
        pred = model(batch_x)       # Forward Pass
        loss = criterion(pred, batch_y) # Calculate Error

        loss.backward()             # Backpropagation: Calculate gradients
        optimizer.step()            # Update weights based on AdamW logic

        train_loss += loss.item() * batch_x.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation Phase: Check if the model is actually learning or just memorizing
    model.eval() # Disable Dropout/BatchNorm noise
    val_loss = 0.0
    with torch.no_grad(): # Disable gradient calculation (saves memory/time)
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred = model(batch_x)
            val_loss += criterion(pred, batch_y).item() * batch_x.size(0)

    val_loss /= len(val_loader.dataset)

    # Checkpointing: Save the model ONLY if it performs better on Validation data
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt") # Save "Best" version

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

# ─── 5. FINAL EVALUATION ──────────────────────────────────────────
# Load the best version (not necessarily the last one)
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
test_loss = 0.0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        pred = model(batch_x)
        test_loss += criterion(pred, batch_y).item() * batch_x.size(0)

test_loss /= len(test_loader.dataset)
print(f"\nFinal Results:\nTest MSE: {test_loss:.4f}")
print(f"Test RMSE: {np.sqrt(test_loss):.4f} (Avg error in $100k units)")

Epoch  20 | Train MSE: 0.3127 | Val MSE: 0.3306
Epoch  40 | Train MSE: 0.2883 | Val MSE: 0.2901
Epoch  60 | Train MSE: 0.2873 | Val MSE: 0.2698
Epoch  80 | Train MSE: 0.2739 | Val MSE: 0.2897
Epoch 100 | Train MSE: 0.2583 | Val MSE: 0.3448
Epoch 120 | Train MSE: 0.2589 | Val MSE: 0.3505

Final Results:
Test MSE: 0.2492
Test RMSE: 0.4992 (Avg error in $100k units)
